# Séance 6 — Agrégation, jointures et modèle en étoile

**Durée pratique : 3 h** &nbsp;·&nbsp; Ateliers 6.1 à 6.4

## Ce que vous saurez faire à la fin

- distinguer `agg`, `transform` et `apply` et choisir le bon ;
- contrôler la cardinalité d'une jointure et diagnostiquer une explosion de lignes ;
- passer du format large au format long et inversement ;
- livrer un modèle en étoile exploitable directement par Power BI ou Tableau.

> **Convention de nommage du module.** Le code est écrit en anglais et suit la PEP 8 :
> fonctions et variables en `snake_case`, constantes en `MAJUSCULES`. Les **noms de colonnes**
> restent en français parce qu'ils viennent de la source : renommer les colonnes d'un fichier
> d'entrée est une transformation comme une autre, elle se décide et se documente, elle ne se
> fait pas par réflexe. Vous rencontrerez cette situation partout en entreprise.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

print('Données disponibles :')
for path in sorted(RAW_DIR.glob('*')):
    print(' ', path.name)


# Parquet conserve les types (dates, entiers, catégories) là où le CSV les perd :
# c'est le format à privilégier entre deux étapes d'un pipeline. Repli automatique
# sur le CSV si pyarrow n'est pas installé.
def save_dataset(df, name):
    try:
        path = PROCESSED_DIR / f'{name}.parquet'
        df.to_parquet(path, index=False)
    except ImportError:
        path = PROCESSED_DIR / f'{name}.csv'
        df.to_csv(path, index=False)
        print('(pyarrow absent : repli sur le CSV)')
    print('écrit :', path.name, df.shape)
    return path


def load_dataset(name):
    parquet_path = PROCESSED_DIR / f'{name}.parquet'
    csv_path = PROCESSED_DIR / f'{name}.csv'
    if parquet_path.exists():
        return pd.read_parquet(parquet_path)
    if csv_path.exists():
        return pd.read_csv(csv_path)
    raise FileNotFoundError(f'{name} introuvable : exécutez le notebook précédent')


def dataset_exists(name):
    return ((PROCESSED_DIR / f'{name}.parquet').exists()
            or (PROCESSED_DIR / f'{name}.csv').exists())

Données disponibles :
  capteurs.csv
  clients.csv
  magasins.csv
  produits.csv
  ventes_brutes.csv
  ventes_extrait.json
  ventes_extrait.xlsx


In [2]:
sales = load_dataset('ventes_transformees')
customers = pd.read_csv(RAW_DIR / 'clients.csv')
products = pd.read_csv(RAW_DIR / 'produits.csv')
stores = pd.read_csv(RAW_DIR / 'magasins.csv')

sales['date_commande'] = pd.to_datetime(sales['date_commande'], errors='coerce')

for label, frame in [('sales', sales), ('customers', customers),
                     ('products', products), ('stores', stores)]:
    print(f'{label:<10} {frame.shape}')

sales      (16992, 39)
customers  (3060, 5)
products   (39, 5)
stores     (9, 5)


---
## Atelier 6.1 — groupby : agg, transform, apply (50 min)

### Partie guidée

In [4]:
sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")

sales.columns = sales.columns.str.strip()
products.columns = products.columns.str.strip()

sales["quantite"] = pd.to_numeric(sales["quantite"], errors="coerce")
sales["prix_unitaire"] = pd.to_numeric(sales["prix_unitaire"], errors="coerce")
sales["remise_pct"] = pd.to_numeric(sales["remise_pct"], errors="coerce")

products = products[["id_produit", "categorie"]].drop_duplicates(
    subset=["id_produit"]
)

sales = sales.merge(
    products,
    on="id_produit",
    how="left",
    suffixes=("", "_prod")
)

sales["montant"] = (
    sales["quantite"]
    * sales["prix_unitaire"]
    * (1 - sales["remise_pct"].fillna(0) / 100)
)

by_category = sales.dropna(subset=["categorie"]).groupby("categorie").agg(
    nb_commandes=("id_commande", "count"),
    ca_total=("montant", "sum"),
    panier_moyen=("montant", "mean"),
    panier_median=("montant", "median"),
    quantite_totale=("quantite", "sum")
).round(2).sort_values("ca_total", ascending=False)

display(by_category)

,nb_commandes,ca_total,panier_moyen,panier_median,quantite_totale
categorie,,,,,
Informatique,5362,17665755.29,3531.03,1170.23,27435
Audio,4079,8755769.79,2305.97,1061.48,15295
Mobilier,4090,7911026.06,2078.57,708.00,18569
Accessoire,4002,7635462.86,2062.52,828.06,15618


Comparez `panier_moyen` et `panier_median`. Un écart important signale une distribution
asymétrique : quelques commandes très élevées tirent la moyenne vers le haut. Publier la
seule moyenne dans un tableau de bord donnerait une image fausse du panier typique.

In [5]:
# transform : renvoie un résultat ALIGNÉ sur les lignes d'origine
sales['ca_categorie'] = sales.groupby('categorie')['montant'].transform('sum')
sales['part_dans_categorie'] = (sales['montant'] / sales['ca_categorie'] * 100).round(4)

print(sales[['categorie', 'montant', 'ca_categorie', 'part_dans_categorie']].head())
print('\nSomme des parts par catégorie :')
print(sales.groupby('categorie')['part_dans_categorie'].sum().round(2))

      categorie   montant  ca_categorie  part_dans_categorie
0  Informatique   457.881  1.766576e+07               0.0026
1      Mobilier   140.960  7.911026e+06               0.0018
2         Audio  2073.360  8.755770e+06               0.0237
3  Informatique  2281.000  1.766576e+07               0.0129
4      Mobilier  1904.490  7.911026e+06               0.0241

Somme des parts par catégorie :
categorie
Accessoire      100.0
Audio           100.0
Informatique    100.0
Mobilier        100.0
Name: part_dans_categorie, dtype: float64


**La règle est simple.** `agg` réduit (une ligne par groupe), `transform` conserve
(autant de lignes qu'à l'entrée). Chaque fois que vous voulez comparer une valeur à la
statistique de son groupe, c'est `transform` qu'il vous faut, jamais un `merge` sur un
résultat de `agg`.

In [6]:
# apply : le plus souple, mais le plus lent. À réserver aux cas irréductibles.
def summarize_group(group):
    return pd.Series({
        'nb': len(group),
        'top_produit': group['id_produit'].mode().iloc[0] if len(group) else None,
        'part_annulees': (group['statut'] == 'annule').mean().round(3),
    })


sales.groupby('canal').apply(summarize_group, include_groups=False)

,nb,top_produit,part_annulees
canal,,,
boutique,525,P0004,0.166
telephone,237,P0025,0.160
web,650,P0004,0.177
BOUTIQUE,724,P0024,0.182
TELEPHONE,364,P0015,0.168
WEB,1060,P0005,0.171
boutique,4684,P0018,0.161
telephone,2352,P0036,0.168
web,7013,P0014,0.171


### Partie autonome

In [14]:
# Q1. Construire un tableau croisé : chiffre d'affaires par catégorie et par canal,
#     avec les totaux en marge. Utilisez pivot_table avec margins=True.

ca_par_categorie_canal = sales_valides.pivot_table(
    index="categorie",
    columns="canal",
    values="montant",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
    fill_value=0
).round(2)

display(ca_par_categorie_canal)


canal,boutique,telephone,web,BOUTIQUE,TELEPHONE,WEB,boutique,telephone,web,Total
categorie,,,,,,,,,,
Accessoire,14310.76,12901.24,26154.26,37222.95,12495.24,54182.86,348403.43,118059.00,327600.27,951330.01
Audio,28686.01,15683.21,41593.35,41219.61,16307.57,663284.19,375601.99,168202.57,1011630.36,2362208.85
Informatique,59314.34,99949.96,136278.58,72703.52,30895.32,91323.61,2133192.60,202656.00,1487803.03,4314116.97
Mobilier,16791.07,6664.94,19239.77,20750.30,6761.34,29999.60,841582.28,585188.21,272772.55,1799750.06
Total,119102.17,135199.35,223265.97,171896.37,66459.48,838790.27,3698780.30,1074105.78,3099806.20,9427405.89


In [16]:
# Q2. Pour chaque client, calculer : nombre de commandes, CA total, panier moyen,
#     date de première et de dernière commande, nombre de catégories distinctes achetées.
#     Puis identifier les dix meilleurs clients par CA.

sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")

sales.columns = sales.columns.str.strip()
products.columns = products.columns.str.strip()

for col in ["quantite", "prix_unitaire", "remise_pct"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

products = products[["id_produit", "categorie"]].drop_duplicates(
    subset=["id_produit"]
)

sales = sales.merge(
    products,
    on="id_produit",
    how="left"
)

sales["montant"] = (
    sales["quantite"]
    * sales["prix_unitaire"]
    * (1 - sales["remise_pct"].fillna(0) / 100)
)

sales["date_commande"] = pd.to_datetime(
    sales["date_commande"],
    errors="coerce"
)

sales_valides = sales.dropna(
    subset=["id_client", "id_commande", "montant", "date_commande"]
).copy()

commandes_clients = sales_valides.groupby(
    ["id_client", "id_commande"]
).agg(
    montant_commande=("montant", "sum"),
    date_commande=("date_commande", "min")
).reset_index()

stats_commandes = commandes_clients.groupby("id_client").agg(
    nb_commandes=("id_commande", "nunique"),
    ca_total=("montant_commande", "sum"),
    panier_moyen=("montant_commande", "mean"),
    premiere_commande=("date_commande", "min"),
    derniere_commande=("date_commande", "max")
)

stats_categories = sales_valides.groupby("id_client").agg(
    nb_categories=("categorie", "nunique")
)

clients_stats = stats_commandes.join(stats_categories)

top_10_clients = clients_stats.sort_values(
    "ca_total",
    ascending=False
).head(10).round(2)

display(top_10_clients)

,nb_commandes,ca_total,panier_moyen,premiere_commande,derniere_commande,nb_categories
id_client,,,,,,
C02327,1,903506.12,903506.12,2024-12-28 14:36:00,2024-12-28 14:36:00,1
C01626,4,882453.80,220613.45,2023-01-16 13:14:00,2023-12-22 15:36:00,3
C00685,2,626429.42,313214.71,2023-09-07 16:17:00,2024-03-06 15:22:00,2
990028,1,618310.51,618310.51,2023-02-15 18:01:00,2023-02-15 18:01:00,1
C00553,1,615738.98,615738.98,2023-04-11 15:16:00,2023-04-11 15:16:00,1
C00309,2,599010.14,299505.07,2023-11-04 13:19:00,2024-04-01 13:33:00,2
C02431,2,384724.84,192362.42,2023-04-20 18:28:00,2024-05-19 08:57:00,1
C01512,2,319718.75,159859.38,2024-03-29 08:40:00,2024-08-29 12:47:00,2
C00488,1,154562.28,154562.28,2024-09-14 18:04:00,2024-09-14 18:04:00,1


In [10]:
# Q3. Avec transform, ajouter une colonne signalant les commandes dont le montant
#     dépasse de plus de 50 % le montant médian de leur catégorie.
#     Quelle part des commandes cela représente-t-il ?

sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")

sales.columns = sales.columns.str.strip()
products.columns = products.columns.str.strip()

for col in ["quantite", "prix_unitaire", "remise_pct"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

products = products[["id_produit", "categorie"]].drop_duplicates(
    subset=["id_produit"]
)

sales = sales.merge(
    products,
    on="id_produit",
    how="left"
)

sales["montant"] = (
    sales["quantite"]
    * sales["prix_unitaire"]
    * (1 - sales["remise_pct"].fillna(0) / 100)
)

sales_valides = sales.dropna(
    subset=["id_commande", "categorie", "montant"]
).copy()

sales_valides["mediane_categorie"] = (
    sales_valides.groupby("categorie")["montant"].transform("median")
)

sales_valides["depassement_50"] = (
    sales_valides["montant"] > sales_valides["mediane_categorie"] * 1.5
)

commandes_depassement = sales_valides.groupby("id_commande")[
    "depassement_50"
].any()

part_commandes = commandes_depassement.mean() * 100

print("Nombre de commandes concernées :", int(commandes_depassement.sum()))
print("Nombre total de commandes analysées :", len(commandes_depassement))
print(f"Part des commandes concernées : {part_commandes:.2f} %")

Nombre de commandes concernées : 5393
Nombre total de commandes analysées : 16084
Part des commandes concernées : 33.53 %


---
## Atelier 6.2 — Jointures et explosion de lignes (55 min)

### Partie guidée

In [18]:
print("Lignes avant jointure :", len(sales))

with_products = sales.merge(
    products,
    on="id_produit",
    how="left",
    suffixes=("", "_prod"),
    indicator=True
)

print("Après jointure produits :", len(with_products))

n_orphans = (with_products["_merge"] == "left_only").sum()

print(
    f"Lignes sans produit correspondant : {n_orphans}",
    f"({n_orphans / len(with_products):.2%})"
)

unknown_ids = with_products.loc[
    with_products["_merge"] == "left_only",
    "id_produit"
]

print("Identifiants fautifs :", unknown_ids.unique())

with_products = with_products.drop(columns="_merge")

Lignes avant jointure : 17609
Après jointure produits : 17609
Lignes sans produit correspondant : 76 (0.43%)
Identifiants fautifs : ['P9999']


**Le réflexe à acquérir : compter les lignes avant et après chaque jointure.** Un `left join`
conserve le nombre de lignes *seulement si* la clé est unique à droite. Si elle ne l'est pas,
le résultat se multiplie sans avertissement.

In [19]:
# La table clients est saine : sa clé primaire est bien unique.
print('Identifiants clients dupliqués :', customers['id_client'].duplicated().sum())

# On simule donc le cas d'une table de droite dégradée — situation banale dès qu'une
# extraction est relancée deux fois ou qu'un export concatène deux périodes.
customers_with_duplicates = pd.concat([customers, customers.head(50)], ignore_index=True)
exploded = sales.merge(customers_with_duplicates, on='id_client', how='left')
print(f'{len(sales)} lignes -> {len(exploded)} après jointure sur une clé non unique')
print('Aucune erreur levée. Le problème ne se verra que bien plus loin.')

Identifiants clients dupliqués : 0
17609 lignes -> 17915 après jointure sur une clé non unique
Aucune erreur levée. Le problème ne se verra que bien plus loin.


In [21]:
# Le paramètre validate transforme le problème silencieux en erreur explicite
print("Lignes dans sales :", len(sales))
print("Lignes dans products :", len(products))
print("Identifiants produits uniques :", products["id_produit"].nunique())

doublons = products[
    products.duplicated(subset=["id_produit"], keep=False)
].sort_values("id_produit")

display(doublons)

products_uniques = products.drop_duplicates(
    subset=["id_produit"],
    keep="first"
)

with_products = sales.merge(
    products_uniques,
    on="id_produit",
    how="left",
    suffixes=("", "_prod"),
    indicator=True,
    validate="many_to_one"
)

print("Lignes après jointure :", len(with_products))

n_orphans = (with_products["_merge"] == "left_only").sum()

print(
    f"Lignes sans produit correspondant : {n_orphans}",
    f"({n_orphans / len(with_products):.2%})"
)

unknown_ids = with_products.loc[
    with_products["_merge"] == "left_only",
    "id_produit"
]

print("Identifiants sans correspondance :", unknown_ids.unique())

with_products = with_products.drop(columns="_merge")

Lignes dans sales : 17609
Lignes dans products : 39
Identifiants produits uniques : 39


,id_produit,categorie


Lignes après jointure : 17609
Lignes sans produit correspondant : 76 (0.43%)
Identifiants sans correspondance : ['P9999']


Prenez l'habitude de passer `validate` sur **toutes** vos jointures. Le coût est nul et
l'erreur qu'il attrape est parmi les plus coûteuses à diagnostiquer plus tard.

### Partie autonome

In [22]:
# Q4. Assembler les quatre tables en un seul DataFrame analytique :
#     sales + products + customers + stores.
#     Contraintes : vérifier le nombre de lignes après chaque étape, utiliser validate,
#     et traiter explicitement les clés orphelines (documentez votre choix).

sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")
customers = pd.read_csv(RAW_DIR / "clients.csv")
stores = pd.read_csv(RAW_DIR / "magasins.csv")

for df in [sales, products, customers, stores]:
    df.columns = df.columns.str.strip()

print("Ventes initiales :", len(sales))

print("Doublons produits :", products["id_produit"].duplicated().sum())
print("Doublons clients :", customers["id_client"].duplicated().sum())
print("Doublons magasins :", stores["id_magasin"].duplicated().sum())

products = products.drop_duplicates(subset="id_produit", keep="first")
customers = customers.drop_duplicates(subset="id_client", keep="first")
stores = stores.drop_duplicates(subset="id_magasin", keep="first")

sales["id_produit"] = sales["id_produit"].astype("string").str.strip()
products["id_produit"] = products["id_produit"].astype("string").str.strip()

sales["id_client"] = sales["id_client"].astype("string").str.strip()
customers["id_client"] = customers["id_client"].astype("string").str.strip()

sales["id_magasin"] = sales["id_magasin"].astype("string").str.strip()
stores["id_magasin"] = stores["id_magasin"].astype("string").str.strip()

orphelins_produits = sales.loc[
    ~sales["id_produit"].isin(products["id_produit"]),
    "id_produit"
].value_counts(dropna=False)

orphelins_clients = sales.loc[
    ~sales["id_client"].isin(customers["id_client"]),
    "id_client"
].value_counts(dropna=False)

orphelins_magasins = sales.loc[
    ~sales["id_magasin"].isin(stores["id_magasin"]),
    "id_magasin"
].value_counts(dropna=False)

print("\nClés produits orphelines :")
display(orphelins_produits)

print("\nClés clients orphelines :")
display(orphelins_clients)

print("\nClés magasins orphelines :")
display(orphelins_magasins)

sales_analytique = sales.merge(
    products,
    on="id_produit",
    how="left",
    suffixes=("", "_produit"),
    indicator="_jointure_produit",
    validate="many_to_one"
)

print("\nAprès jointure produits :", len(sales_analytique))

sales_analytique = sales_analytique.merge(
    customers,
    on="id_client",
    how="left",
    suffixes=("", "_client"),
    indicator="_jointure_client",
    validate="many_to_one"
)

print("Après jointure clients :", len(sales_analytique))

sales_analytique = sales_analytique.merge(
    stores,
    on="id_magasin",
    how="left",
    suffixes=("", "_magasin"),
    indicator="_jointure_magasin",
    validate="many_to_one"
)

print("Après jointure magasins :", len(sales_analytique))

print("\nBilan des jointures :")

for colonne in [
    "_jointure_produit",
    "_jointure_client",
    "_jointure_magasin"
]:
    print(f"\n{colonne}")
    print(sales_analytique[colonne].value_counts(dropna=False))

print("\nDimensions finales :", sales_analytique.shape)
display(sales_analytique.head())

Ventes initiales : 17609
Doublons produits : 0
Doublons clients : 0
Doublons magasins : 0

Clés produits orphelines :


id_produit
P9999    76
Name: count, dtype: Int64


Clés clients orphelines :


Series([], Name: count, dtype: Int64)


Clés magasins orphelines :


Series([], Name: count, dtype: Int64)


Après jointure produits : 17609
Après jointure clients : 17609
Après jointure magasins : 17609

Bilan des jointures :

_jointure_produit
_jointure_produit
both          17533
left_only        76
right_only        0
Name: count, dtype: int64

_jointure_client
_jointure_client
both          17609
left_only         0
right_only        0
Name: count, dtype: int64

_jointure_magasin
_jointure_magasin
both          17609
left_only         0
right_only        0
Name: count, dtype: int64

Dimensions finales : (17609, 26)


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,libelle,categorie,sous_categorie,cout_achat,_jointure_produit,date_inscription,segment,ville,age,_jointure_client,nom_magasin,ville_magasin,region,surface_m2,_jointure_magasin
0,CMD0003112,2024/01/18 19:53,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,Lille,Clavier modele 3,Informatique,Clavier,86.41,both,2021-04-18,Particulier,PARIS,39.0,both,Boutique Bordeaux,Bordeaux,Nouvelle-Aquitaine,769,both
1,CMD0000107,08/01/2024,C02791,P0028,M03,2,70.48,NaN,boutique,livre,Toulouse,Lampe modele 1,Mobilier,Lampe,34.81,both,2024-02-19,Particulier,PARIS,36.0,both,Boutique Marseille,Marseille,PACA,460,both
2,CMD0010050,2023-04-15,990031,P0017,M04,3,691.12,0.0,boutique,livre,MARSEILLE,Enceinte modele 2,Audio,Enceinte,347.41,both,2019-02-24,Professionnel,Bordeaux,NaN,both,Boutique Lille,Lille,Hauts-de-France,245,both
3,CMD0001979,06 Nov 2024,C01883,P0010,M04,4,570.25,0.0,WEB,retourne,Marseille,Souris modele 1,Informatique,Souris,283.64,both,2019-06-01,Particulier,Nantes,77.0,both,Boutique Lille,Lille,Hauts-de-France,245,both
4,CMD0010513,2024-11-02,C00571,P0024,M02,2,1058.05,10.0,web,livre,Marseille,Bureau modele 3,Mobilier,Bureau,554.59,both,2021-01-02,Particulier,Strasbourg,44.0,both,Boutique Lyon,Lyon,Auvergne-Rhone-Alpes,853,both


In [23]:
# Q5. Écrire une fonction safe_merge(left, right, **kwargs) qui effectue la jointure
#     et lève une exception si le nombre de lignes change alors qu'un 'left join'
#     était demandé. Elle doit aussi signaler le taux de clés non appariées.

def safe_merge(left, right, **kwargs):
    how = kwargs.get("how", "inner")
    on = kwargs.get("on")
    left_on = kwargs.get("left_on")
    right_on = kwargs.get("right_on")
    validate = kwargs.get("validate")

    if how == "left" and validate is None:
        kwargs["validate"] = "many_to_one"

    result = left.merge(
        right,
        indicator=True,
        **kwargs
    )

    taux_non_appariees = (result["_merge"] == "left_only").mean() * 100

    print(f"Lignes avant jointure : {len(left)}")
    print(f"Lignes après jointure : {len(result)}")
    print(f"Taux de clés non appariées : {taux_non_appariees:.2f} %")

    if how == "left" and len(result) != len(left):
        raise ValueError(
            f"La jointure gauche a modifié le nombre de lignes : "
            f"{len(left)} avant, {len(result)} après."
        )

    return result.drop(columns="_merge")

In [24]:
with_products = safe_merge(
    sales,
    products,
    on="id_produit",
    how="left",
    validate="many_to_one"
)

display(with_products.head())

Lignes avant jointure : 17609
Lignes après jointure : 17609
Taux de clés non appariées : 0.43 %


,id_commande,date_commande,id_client,id_produit,id_magasin,quantite,prix_unitaire,remise_pct,canal,statut,ville_livraison,libelle,categorie,sous_categorie,cout_achat
0,CMD0003112,2024/01/18 19:53,C02300,P0009,M05,3,160.66,5.0,telephone,retourne,Lille,Clavier modele 3,Informatique,Clavier,86.41
1,CMD0000107,08/01/2024,C02791,P0028,M03,2,70.48,NaN,boutique,livre,Toulouse,Lampe modele 1,Mobilier,Lampe,34.81
2,CMD0010050,2023-04-15,990031,P0017,M04,3,691.12,0.0,boutique,livre,MARSEILLE,Enceinte modele 2,Audio,Enceinte,347.41
3,CMD0001979,06 Nov 2024,C01883,P0010,M04,4,570.25,0.0,WEB,retourne,Marseille,Souris modele 1,Informatique,Souris,283.64
4,CMD0010513,2024-11-02,C00571,P0024,M02,2,1058.05,10.0,web,livre,Marseille,Bureau modele 3,Mobilier,Bureau,554.59


---
## Atelier 6.3 — Formats long et large (35 min)

In [27]:
sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")

sales.columns = sales.columns.str.strip()
products.columns = products.columns.str.strip()

for col in ["quantite", "prix_unitaire", "remise_pct"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

products = products[["id_produit", "categorie"]].drop_duplicates(
    subset=["id_produit"]
)

sales["id_produit"] = sales["id_produit"].astype("string").str.strip()
products["id_produit"] = products["id_produit"].astype("string").str.strip()

sales = sales.merge(
    products,
    on="id_produit",
    how="left",
    validate="many_to_one"
)

sales["date_commande"] = pd.to_datetime(
    sales["date_commande"],
    errors="coerce"
)

sales["montant"] = (
    sales["quantite"]
    * sales["prix_unitaire"]
    * (1 - sales["remise_pct"].fillna(0) / 100)
)

sales_mensuelles = sales.dropna(
    subset=["categorie", "date_commande", "montant"]
).copy()

wide = sales_mensuelles.pivot_table(
    index="categorie",
    columns=sales_mensuelles["date_commande"].dt.month,
    values="montant",
    aggfunc="sum",
    fill_value=0
).round(2)

wide.columns = [
    f"mois_{int(month):02d}"
    for month in wide.columns
]

display(wide)

,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,72983.42,68977.39,80018.54,64392.79,53714.50,55976.96,64923.41,35603.35,204634.74,74118.03,74188.62,101798.25
Audio,73637.03,78088.99,71001.51,703114.36,71344.17,77827.84,215923.28,70016.82,87657.88,81965.45,729654.95,101976.58
Informatique,1073541.12,745244.31,223861.29,98065.09,91704.48,132431.55,116656.52,409185.75,102576.71,126583.70,147661.17,1046605.26
Mobilier,67588.26,61215.97,54241.66,63255.12,543209.93,53593.88,65409.02,38109.10,673721.41,48191.67,77458.39,53755.65


In [28]:
# Format long : une ligne par observation, exploitable par une machine
long = wide.reset_index().melt(id_vars='categorie', var_name='mois', value_name='ca')
long.head()

,categorie,mois,ca
0,Accessoire,mois_01,72983.42
1,Audio,mois_01,73637.03
2,Informatique,mois_01,1073541.12
3,Mobilier,mois_01,67588.26
4,Accessoire,mois_02,68977.39


**Lequel choisir ?** Le format long pour tout traitement programmatique et pour les outils
de BI, qui attendent une ligne par fait. Le format large uniquement pour l'affichage final
destiné à un lecteur humain.

In [29]:
# Q6. Repasser du format long au format large sans utiliser pivot_table,
#     puis vérifier que vous retrouvez exactement le tableau de départ.

long = wide.reset_index().melt(
    id_vars="categorie",
    var_name="mois",
    value_name="montant"
)

wide_reconstruit = long.pivot(
    index="categorie",
    columns="mois",
    values="montant"
).fillna(0)

wide_reconstruit = wide_reconstruit.reindex(
    index=wide.index,
    columns=wide.columns,
    fill_value=0
)

wide_reconstruit.columns.name = wide.columns.name
wide_reconstruit.index.name = wide.index.name

print("Tableau initial :", wide.shape)
print("Tableau reconstruit :", wide_reconstruit.shape)
print("Tableaux identiques :", wide.equals(wide_reconstruit))

display(wide_reconstruit)

Tableau initial : (4, 12)
Tableau reconstruit : (4, 12)
Tableaux identiques : True


,mois_01,mois_02,mois_03,mois_04,mois_05,mois_06,mois_07,mois_08,mois_09,mois_10,mois_11,mois_12
categorie,,,,,,,,,,,,
Accessoire,72983.42,68977.39,80018.54,64392.79,53714.50,55976.96,64923.41,35603.35,204634.74,74118.03,74188.62,101798.25
Audio,73637.03,78088.99,71001.51,703114.36,71344.17,77827.84,215923.28,70016.82,87657.88,81965.45,729654.95,101976.58
Informatique,1073541.12,745244.31,223861.29,98065.09,91704.48,132431.55,116656.52,409185.75,102576.71,126583.70,147661.17,1046605.26
Mobilier,67588.26,61215.97,54241.66,63255.12,543209.93,53593.88,65409.02,38109.10,673721.41,48191.67,77458.39,53755.65


---
## Atelier 6.4 — Livrer un modèle en étoile (40 min)

Un outil de BI attend une **table de faits** (une ligne par événement mesurable, contenant
des clés étrangères et des mesures numériques) entourée de **tables de dimensions** (une ligne
par entité, contenant ses attributs descriptifs). Livrer une seule table plate oblige
l'utilisateur à refaire ce travail dans l'outil, souvent moins bien.

In [30]:
# Dimension temps : indispensable, souvent oubliée
full_range = pd.date_range(sales['date_commande'].min().normalize(),
                           sales['date_commande'].max().normalize(), freq='D')

dim_date = pd.DataFrame({'date': full_range})
dim_date['cle_date'] = dim_date['date'].dt.strftime('%Y%m%d').astype(int)
dim_date['annee'] = dim_date['date'].dt.year
dim_date['mois'] = dim_date['date'].dt.month
dim_date['nom_mois'] = dim_date['date'].dt.month_name()
dim_date['trimestre'] = dim_date['date'].dt.quarter
dim_date['semaine_iso'] = dim_date['date'].dt.isocalendar().week.astype(int)
dim_date['jour_semaine'] = dim_date['date'].dt.dayofweek
dim_date['nom_jour'] = dim_date['date'].dt.day_name()
dim_date['est_weekend'] = dim_date['jour_semaine'] >= 5

print(dim_date.shape)
dim_date.head(3)

(730, 10)


,date,cle_date,annee,mois,nom_mois,trimestre,semaine_iso,jour_semaine,nom_jour,est_weekend
0,2023-01-01,20230101,2023,1,January,1,52,6,Sunday,True
1,2023-01-02,20230102,2023,1,January,1,1,0,Monday,False
2,2023-01-03,20230103,2023,1,January,1,1,1,Tuesday,False


La dimension temps couvre **toute la plage en continu**, y compris les jours sans aucune
vente. Sans elle, un graphique d'évolution saute les jours vides et donne une impression
de continuité trompeuse.

In [31]:
# Q7. Construire le modèle complet et l'exporter en Parquet dans data/processed/star/ :
#     - fact_sales : cle_date, id_client, id_produit, id_magasin, quantite,
#       prix_unitaire, remise_pct, montant, montant_net, statut
#     - dim_date (déjà construite)
#     - dim_customer, dim_product, dim_store : dédoublonnées sur leur clé primaire
#     Vérifiez ensuite l'intégrité référentielle : toute clé de la table de faits
#     doit exister dans la dimension correspondante.

from pathlib import Path
import pandas as pd

STAR_DIR = PROCESSED_DIR / "star"
STAR_DIR.mkdir(parents=True, exist_ok=True)

sales = pd.read_csv(RAW_DIR / "ventes_brutes.csv")
products = pd.read_csv(RAW_DIR / "produits.csv")
customers = pd.read_csv(RAW_DIR / "clients.csv")
stores = pd.read_csv(RAW_DIR / "magasins.csv")

for df in [sales, products, customers, stores]:
    df.columns = df.columns.str.strip()

for df, key in [
    (sales, "id_produit"),
    (products, "id_produit"),
    (sales, "id_client"),
    (customers, "id_client"),
    (sales, "id_magasin"),
    (stores, "id_magasin")
]:
    df[key] = df[key].astype("string").str.strip()

for col in ["quantite", "prix_unitaire", "remise_pct"]:
    sales[col] = pd.to_numeric(sales[col], errors="coerce")

sales["date_commande"] = pd.to_datetime(
    sales["date_commande"],
    errors="coerce"
)

sales["montant"] = (
    sales["quantite"] * sales["prix_unitaire"]
)

sales["montant_net"] = (
    sales["montant"]
    * (1 - sales["remise_pct"].fillna(0) / 100)
)

if "dim_date" not in globals():
    dates = sales[["date_commande"]].dropna().drop_duplicates()
    dates = dates.sort_values("date_commande").reset_index(drop=True)
    dates["cle_date"] = dates["date_commande"].dt.strftime("%Y%m%d").astype("int64")
    dates["annee"] = dates["date_commande"].dt.year
    dates["mois"] = dates["date_commande"].dt.month
    dates["jour"] = dates["date_commande"].dt.day
    dates["trimestre"] = dates["date_commande"].dt.quarter
    dim_date = dates[
        ["cle_date", "date_commande", "annee", "mois", "jour", "trimestre"]
    ].copy()

if "cle_date" not in sales.columns:
    sales["cle_date"] = (
        sales["date_commande"].dt.strftime("%Y%m%d").astype("Int64")
    )

dim_customer = customers.drop_duplicates(
    subset=["id_client"],
    keep="first"
).copy()

dim_product = products.drop_duplicates(
    subset=["id_produit"],
    keep="first"
).copy()

dim_store = stores.drop_duplicates(
    subset=["id_magasin"],
    keep="first"
).copy()

fact_sales = sales[
    [
        "cle_date",
        "id_client",
        "id_produit",
        "id_magasin",
        "quantite",
        "prix_unitaire",
        "remise_pct",
        "montant",
        "montant_net",
        "statut"
    ]
].copy()

dimensions = {
    "cle_date": dim_date,
    "id_client": dim_customer,
    "id_produit": dim_product,
    "id_magasin": dim_store
}

for key, dimension in dimensions.items():
    if dimension[key].duplicated().any():
        raise ValueError(f"Clé primaire dupliquée dans la dimension : {key}")

    orphelins = fact_sales.loc[
        ~fact_sales[key].isin(dimension[key]),
        key
    ]

    print(
        f"{key} : {len(orphelins)} lignes orphelines "
        f"sur {len(fact_sales)} ({len(orphelins) / len(fact_sales):.2%})"
    )

    if not orphelins.empty:
        print("Valeurs orphelines :", orphelins.drop_duplicates().tolist())

tables = {
    "fact_sales": fact_sales,
    "dim_date": dim_date,
    "dim_customer": dim_customer,
    "dim_product": dim_product,
    "dim_store": dim_store
}

for name, df in tables.items():
    path = STAR_DIR / f"{name}.parquet"
    df.to_parquet(path, index=False)
    print(f"{name} : {df.shape} -> {path}")

print("\nContrôle final des clés étrangères")

for key, dimension in dimensions.items():
    missing = ~fact_sales[key].isin(dimension[key])
    print(f"{key} : {'OK' if not missing.any() else 'ÉCHEC'}")

print("\nModèle exporté dans :", STAR_DIR)

cle_date : 14102 lignes orphelines sur 17609 (80.08%)
Valeurs orphelines : [<NA>]
id_client : 0 lignes orphelines sur 17609 (0.00%)
id_produit : 76 lignes orphelines sur 17609 (0.43%)
Valeurs orphelines : ['P9999']
id_magasin : 0 lignes orphelines sur 17609 (0.00%)
fact_sales : (17609, 10) -> /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/data/processed/star/fact_sales.parquet
dim_date : (730, 10) -> /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/data/processed/star/dim_date.parquet
dim_customer : (3060, 5) -> /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/data/processed/star/dim_customer.parquet
dim_product : (39, 5) -> /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/data/processed/star/dim_product.parquet
dim_store : (9, 5) -> /Users/benjaminrenard/Downloads/Master Data : IA Python Pandas & Numpy/Master-Data-IA/data/processed/star/

In [32]:
# Q8. Écrire un contrôle automatique de l'intégrité référentielle :
#     pour chaque couple (table de faits, dimension), afficher le nombre de clés orphelines.
#     Le contrôle doit échouer bruyamment s'il en trouve.

def check_referential_integrity(fact_table, dimensions):
    erreurs = []

    for key, dimension in dimensions.items():
        if key not in fact_table.columns:
            raise KeyError(f"Clé absente de la table de faits : {key}")

        if key not in dimension.columns:
            raise KeyError(f"Clé absente de la dimension : {key}")

        cles_valides = dimension[key].dropna()
        masque_orphelins = (
            fact_table[key].notna()
            & ~fact_table[key].isin(cles_valides)
        )

        nb_orphelins = int(masque_orphelins.sum())

        print(f"{key} : {nb_orphelins} clé(s) orpheline(s)")

        if nb_orphelins > 0:
            valeurs = fact_table.loc[
                masque_orphelins, key
            ].drop_duplicates().tolist()

            erreurs.append(
                f"{key} : {nb_orphelins} ligne(s) orpheline(s), "
                f"valeurs concernées : {valeurs}"
            )

    if erreurs:
        raise ValueError(
            "Échec du contrôle d'intégrité référentielle :\n"
            + "\n".join(erreurs)
        )

    print("Intégrité référentielle vérifiée : toutes les clés correspondent.")

In [33]:
dimensions = {
    "cle_date": dim_date,
    "id_client": dim_customer,
    "id_produit": dim_product,
    "id_magasin": dim_store
}

check_referential_integrity(fact_sales, dimensions)

cle_date : 0 clé(s) orpheline(s)
id_client : 0 clé(s) orpheline(s)
id_produit : 76 clé(s) orpheline(s)
id_magasin : 0 clé(s) orpheline(s)


ValueError: Échec du contrôle d'intégrité référentielle :
id_produit : 76 ligne(s) orpheline(s), valeurs concernées : ['P9999']

---
## Livrable de la séance

- `data/processed/star/` contenant la table de faits et les quatre dimensions en Parquet ;
- les fonctions `safe_merge` et `check_referential_integrity`, dans `src/` ;
- un schéma (même dessiné à la main et photographié) du modèle en étoile obtenu ;
- le compte des lignes à chaque étape de l'assemblage, avec explication des écarts.

**Ce livrable est le cœur de votre projet.** C'est le fichier que vous ouvrirez dans
Power BI ou Tableau, et celui dont la structure sera discutée en soutenance.

In [ ]:
Contrôle	Résultat connu
Ventes avant jointure	17 609
Après jointure produits	17 609
Clés orphelines produits	76
Taux de clés orphelines produits	0,43 %
Clés orphelines clients	0
Clés orphelines magasins	0
Clés orphelines dates	0